# Day 2 extras: (A) speed of every method on ONE GPU, (B) GSAD on 200 ExDark images

**Settings:** Accelerator = GPU T4 (x2 is fine; one is used), Internet = ON. **Save Version → Save & Run All**. ~1.5 h.
At the end, download **`day2_extras.zip`** from the Output tab.

* **A — speed.** Speed must be compared on the same hardware. Every method is timed here on the same T4,
  on the same 100 LOL-v2-real images (600×400): only the network is timed, GPU synchronised, first image skipped
  (warm-up). Results go to `results/speed.csv` with parameter counts.
* **B — GSAD on ExDark.** GSAD needs ~16 s per ExDark image, so it runs on a fixed 200-image subset
  (`results/exdark_subset200.txt` in the repo). We rebuild the ExDark images with OUR `exdark_to_yolo.py` and check
  every image's pixel fingerprint (SHA-1) against the list made on the Mac, so GSAD sees exactly the same pictures
  as the other methods did.

In [ ]:
# 1. GPU + our repo (scripts, subset list) + baseline repos
#    Stop immediately without a GPU: speeds would be CPU speeds and GSAD cannot run at all.
import torch
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4, then run again."
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /kaggle/working
!git clone -q https://github.com/Crazycraftie/low-light-enhancement-btp.git btp
%cd /kaggle/working/btp
!git log --oneline | head -1
!mkdir -p baselines && cd baselines && git clone -q --depth 1 https://github.com/Li-Chongyi/Zero-DCE.git && git clone -q --depth 1 https://github.com/vis-opt-group/SCI.git && git clone -q --depth 1 https://github.com/caiyuanhao1998/Retinexformer.git && git clone -q --depth 1 https://github.com/TaoWangzj/LLFormer.git && git clone -q --depth 1 https://github.com/jinnh/GSAD.git
!pip install -q gdown einops addict future lmdb natsort yacs yapf thop timm lpips kmeans_pytorch tensorboardX

In [ ]:
# 2. Downloads (Google Drive; gdown is sometimes flaky -> retry). Weights + LOL-v2 + ExDark.
import os, subprocess
def gdl(fid, out):
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    for _ in range(4):
        if subprocess.call(f"gdown -q {fid} -O {out}", shell=True) == 0 and os.path.exists(out):
            return
    raise RuntimeError("download failed: " + out)
gdl("1xDwQtTCj3tlAVCTJgYrzonBGVwqeOhKu", "baselines/Retinexformer/pretrained_weights/LOL_v1.pth")
# Fingerprint check: the file MUST be the LOL-v1 model (an earlier version of this notebook downloaded
# LOL_v2_real.pth by mistake - a wrong Drive ID). md5 b6db892d... verified on the Mac (25.15 dB on LOL-v1).
import hashlib
_md5 = hashlib.md5(open("baselines/Retinexformer/pretrained_weights/LOL_v1.pth", "rb").read()).hexdigest()
assert _md5 == "b6db892d02ff410b57fcd311dedf515c", f"WRONG WEIGHTS: md5 {_md5} is not LOL_v1.pth"
print("LOL_v1.pth fingerprint OK:", _md5)
gdl("1nTLY_gChn2_Va2BfS_R_W-bdjsqsLHJB", "baselines/LLFormer/weights_lol/models/model_bestPSNR.pth")
gdl("1Qx4y04tPsKwF_WJypY9HZc7K4BVm4f8H", "baselines/GSAD/checkpoints/lolv1_gen.pth")
gdl("1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U", "/kaggle/tmp/lolv2.zip")
gdl("1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx", "/kaggle/tmp/exdark_images.zip")
gdl("1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i", "/kaggle/tmp/exdark_gt.zip")
!unzip -q -o /kaggle/tmp/lolv2.zip -d data/
!mkdir -p data/ExDark /kaggle/tmp/exd && unzip -q -o /kaggle/tmp/exdark_images.zip -d /kaggle/tmp/exd && unzip -q -o /kaggle/tmp/exdark_gt.zip -d /kaggle/tmp/exd
!mv /kaggle/tmp/exd/ExDark data/ExDark/images && mv /kaggle/tmp/exd/ExDark_Annno data/ExDark/annotations
!curl -sL https://raw.githubusercontent.com/cs-chan/Exclusively-Dark-Image-Dataset/master/Groundtruth/imageclasslist.txt -o data/ExDark/imageclasslist.txt
!echo "LOLv2-real test: $(ls data/LOLv2/Real_captured/Test/Low | wc -l) (100)   ExDark images: $(find data/ExDark/images -type f | grep -v __MACOSX | wc -l) (7363)"

In [ ]:
# 3. PART A - speed on this GPU. Same 100 LOL-v2-real images (600x400) for every method.
#    Enhanced images go to results/<method>/_speed (not needed); each script appends one row to results/speed.csv.
IN = "data/LOLv2/Real_captured/Test/Low"
R = "baselines/Retinexformer"
for m in ["zerodce", "sci", "llformer"]:
    !python scripts/run_light_baselines.py --method {m} --input {IN} --dataset _speed 2>&1 | grep -E "saved|Error|Traceback"
!python scripts/retinexformer_infer.py --repo {R} --opt {R}/Options/RetinexFormer_LOL_v1.yml --weights {R}/pretrained_weights/LOL_v1.pth --input {IN} --output results/retinexformer/_speed --method Retinexformer 2>&1 | grep -E "Mean|params|Error|Traceback"

# GSAD: timed in the GSAD notebook on the same T4 (2.36 s/img on these images, 10 steps). Here: its parameter count.
import torch
sd = torch.load("baselines/GSAD/checkpoints/lolv1_gen.pth", map_location="cpu")
print(f"GSAD denoising network: {sum(v.numel() for v in sd.values()) / 1e6:.2f}M parameters")
!cat results/speed.csv

In [ ]:
# 4. PART B - rebuild the ExDark subset with OUR script, then verify the 200 images pixel-by-pixel.
!python scripts/exdark_to_yolo.py 2>&1 | tail -3
import hashlib, numpy as np
from PIL import Image
rows = [l.split("\t") for l in open("results/exdark_subset200.txt").read().splitlines()[1:]]
bad = [s for s, _, h in rows
       if hashlib.sha1(np.asarray(Image.open(f"data/exdark_yolo/raw/images/{s}.png").convert("RGB")).tobytes()).hexdigest() != h]
print(f"pixel check: {len(rows) - len(bad)}/{len(rows)} images identical to the Mac version", "" if not bad else f"-> MISMATCH: {bad[:5]}")
os.makedirs("/kaggle/tmp/exdark200", exist_ok=True)
for s, _, _ in rows:
    dst = f"/kaggle/tmp/exdark200/{s}.png"
    if not os.path.exists(dst):
        os.symlink(os.path.abspath(f"data/exdark_yolo/raw/images/{s}.png"), dst)
open("results/exdark200_pixel_check.txt", "w").write(f"{len(rows) - len(bad)}/{len(rows)} identical\n" + "\n".join(bad))

In [ ]:
# 5. GSAD on the 200 images: their unpaired script (NO ground-truth trick - there is no ground truth), patched to
#    keep file names, time each image, use the LOL-v1 weights + LOL-v1 sampling schedule (20 steps) - the same
#    training data as the Retinexformer weights we used on ExDark - and fix SSIM for new scikit-image.
%cd /kaggle/working/btp/baselines/GSAD
import json
cfg = json.load(open("config/lolv1_test.json"))
cfg["name"] = "exdark200"
cfg.setdefault("train", {})["manual_seed"] = 4088          # fixed seed: diffusion sampling is random
json.dump(cfg, open("config/exdark200_test.json", "w"), indent=2)

src = open("test_unpaired.py").read()
def patch(old, new):
    global src
    assert src.count(old) == 1, "anchor not found once:\n" + old
    src = src.replace(old, new)
patch("    InputPath = args.input\n",
      "    InputPath = args.input\n    import time, json\n    BTP_OUT = os.environ['BTP_OUT']; os.makedirs(BTP_OUT, exist_ok=True)\n    _times = []\n")
patch("        path = InputPath + Image_names[i]\n", "        path = os.path.join(InputPath, Image_names[i])\n")
patch("        diffusion.test(continous=False)\n",
      "        torch.cuda.synchronize(); _t0 = time.perf_counter()\n"
      "        diffusion.test(continous=False)\n"
      "        torch.cuda.synchronize(); _times.append(time.perf_counter() - _t0)\n"
      "        torch.cuda.empty_cache()\n")
patch("        util.save_img(\n                normal_img, '{}/{}_normal.png'.format(result_path, i+1))\n",
      "        util.save_img(normal_img, os.path.join(BTP_OUT, os.path.splitext(Image_names[i])[0] + '.png'))\n")
patch("\n\nif __name__ == '__main__':",
      "\n    _t = _times[1:] if len(_times) > 1 else _times\n"
      "    json.dump({'n_images': len(_times), 'mean_s_per_image': float(np.mean(_t)), 'steps': opt['model']['beta_schedule']['val']['n_timestep'],\n"
      "               'gpu': torch.cuda.get_device_name(0)}, open(os.path.join(BTP_OUT, 'timing.json'), 'w'))\n"
      "    print(f'BTP timing: {np.mean(_t):.3f} s/image over {len(_t)} images')\n"
      "\n\nif __name__ == '__main__':")
src = src.replace("import natsort\n", "import natsort\nimport numpy as np\n", 1)
open("unpaired_btp.py", "w").write(src)
u = open("utils/util.py").read()
u = u.replace("SSIM(imgA, imgB, full=True, multichannel=True)", "SSIM(imgA, imgB, full=True, channel_axis=-1)").replace("full=True, multichannel=True)", "full=True)")
open("utils/util.py", "w").write(u)
print("patched; running GSAD on 200 images (~1 h) ...")
OUT = "/kaggle/working/btp/results/gsad/ExDark"
!BTP_OUT={OUT} python unpaired_btp.py --dataset config/lolv1.yml --config config/exdark200_test.json --input /kaggle/tmp/exdark200 > /kaggle/working/gsad_exdark.log 2>&1
!grep -E "BTP timing" /kaggle/working/gsad_exdark.log
n = len([f for f in os.listdir(OUT) if f.endswith(".png")]) if os.path.isdir(OUT) else 0
print("saved:", n, "/ 200")
if n != 200:
    !tail -n 40 /kaggle/working/gsad_exdark.log
    raise RuntimeError(f"GSAD ExDark: {n}/200 images - see log above")
%cd /kaggle/working/btp

In [ ]:
# 6. Package for download: speed.csv, GSAD ExDark images + timing, pixel check, log
!cp /kaggle/working/gsad_exdark.log results/gsad/ExDark/ 2>/dev/null
!cd /kaggle/working/btp && zip -q -r /kaggle/working/day2_extras.zip results/speed.csv results/gsad/ExDark results/exdark200_pixel_check.txt
!ls -la /kaggle/working/day2_extras.zip && cat results/gsad/ExDark/timing.json && cat results/exdark200_pixel_check.txt | head -1